# Biology (Class 9-10) — normalisation

Stage 2. Takes the raw NCTB-SchoolText corpus and produces a clean Biology subset fit for
entity/relation extraction. Three repairs, in order of how much they matter:

1. **`ক্ত` → `স্ত` OCR corruption.** The extractor systematically misreads the `ক্ত` conjunct.
   `রক্ত` (blood) — a core entity in the transport chapter — is spelled `রস্ত` in ~20% of its
   occurrences. Left alone this either drops those mentions or invents a phantom node.
2. **Chapter titles.** 6 of 14 stored titles are wrong; they become KG node labels, so they
   are corrected against the printed textbook rather than trusted from the corpus.
3. **Junk chunks.** Page furniture and unrecoverable OCR, per the stage-1 heuristics.

Self-contained: derives the OCR fix list from the corpus itself, no uploads needed.

In [ ]:
import json, glob, re, zipfile, shutil, collections, urllib.request
from pathlib import Path
import pandas as pd

MENDELEY_URL = "https://data.mendeley.com/public-files/datasets/f3882ccczp/files/54e66048-5d2b-47a0-a489-a0b104119476/file_downloaded"
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ZIP_PATH, CORPUS = WORK / "NCTB-SchoolText.zip", WORK / "nctb"
SUBJECT = "biology_secondary"

if not CORPUS.exists():
    if not ZIP_PATH.exists():
        req = urllib.request.Request(MENDELEY_URL, headers={"User-Agent": UA})
        with urllib.request.urlopen(req, timeout=120) as r, open(ZIP_PATH, "wb") as f:
            shutil.copyfileobj(r, f)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(CORPUS)

rows = []
for f in sorted(glob.glob(f"{CORPUS}/classNineTen/processed_chapters_{SUBJECT}/*.jsonl")):
    with open(f, encoding="utf-8") as fh:
        rows += [json.loads(l) for l in fh if l.strip()]

df = pd.DataFrame(rows)
print(f"{len(df)} chunks, {df.chapter_no.nunique()} chapters")

## 1 — `ক্ত` / `স্ত` repair

A blanket `স্ত`→`ক্ত` substitution would be wrong: `স্ত` is a legitimate conjunct in `বস্তু`,
`মস্তিষ্ক`, `স্তর`, `বাস্তুতন্ত্র` and many others. Instead a `স্ত` token is treated as corrupt
only when its `ক্ত` counterpart independently appears elsewhere in the same corpus — the book
spelling the word correctly somewhere else is the evidence.

That rule is conservative: it fixes what it can prove and leaves the rest. Add anything it
misses to `EXTRA_FIX` after reviewing `data/biology_ocr_kta_review.csv`.

In [ ]:
BENGALI_TOKEN = re.compile(r"[ঀ-৿]+")

counts = collections.Counter()
for t in df.text:
    counts.update(BENGALI_TOKEN.findall(t))

# Real Bangla words that happen to have a valid ক্ত-twin, so corroboration alone would
# "correct" them into a different real word. ব্যস্ত (busy) -> ব্যক্ত (expressed) is the
# hazard; it happens not to occur standalone here, but the rule shouldn't depend on that.
BLOCKLIST = {"ব্যস্ত"}

kta_forms = {w for w in counts if "ক্ত" in w}
AUTO_FIX = {w: w.replace("স্ত", "ক্ত")
            for w in counts
            if "স্ত" in w and w not in BLOCKLIST and w.replace("স্ত", "ক্ত") in kta_forms}

# Compounds ending in -শক্তি whose exact corrected form appears nowhere else, so the
# corroboration rule can't prove them. Unambiguous on inspection.
EXTRA_FIX = {
    "সৌরশস্তিকে":  "সৌরশক্তিকে",
    "দৃষ্টিশস্তি":  "দৃষ্টিশক্তি",
    "বাকশস্তি":    "বাকশক্তি",
    "ইচ্ছাশস্তির":  "ইচ্ছাশক্তির",
    "চিন্তাশস্তি":  "চিন্তাশক্তি",
}

FIXES = {**AUTO_FIX, **EXTRA_FIX}

print(f"{len(FIXES)} token repairs ({len(AUTO_FIX)} auto + {len(EXTRA_FIX)} manual) "
      f"covering {sum(counts[w] for w in FIXES)} occurrences")
for w, fix in sorted(FIXES.items(), key=lambda kv: -counts[kv[0]])[:10]:
    print(f"  {w:<18} x{counts[w]:<4} -> {fix}")

In [ ]:
# Replace whole tokens only. A plain substring pass is wrong: the key `রস্ত` also matches
# across the conjunct boundary inside প্রস্তুত (প্র + স্তুত), which silently destroyed every
# occurrence of that word when this was written as a substring substitution.
def repair(text):
    return BENGALI_TOKEN.sub(lambda m: FIXES.get(m.group(), m.group()), text)

df["text_raw"] = df["text"]
df["text"] = df["text"].map(repair)
df["was_repaired"] = df.text != df.text_raw

# Count tokens, not substrings — substring counts move for legitimate reasons (fixing
# রস্তরসের -> রক্তরসের also changes the substring count of স্তর) and produce false alarms.
def token_counts(series):
    c = collections.Counter()
    for t in series:
        c.update(BENGALI_TOKEN.findall(t))
    return c

tb, ta = token_counts(df.text_raw), token_counts(df.text)
print(f"{df.was_repaired.sum()} of {len(df)} chunks changed\n")

print("repaired:")
for w in ["রক্ত", "শক্তি", "অভিব্যক্তি", "জীবপ্রযুক্তি"]:
    print(f"  {w:<16} {tb[w]:>4} -> {ta[w]:>4}")

print("\nlegitimate স্ত words (must not change):")
for w in ["বস্তু", "মস্তিষ্ক", "স্তর", "প্রস্তুত", "স্ত্রী", "বাস্তুতন্ত্র", "ব্যস্ত"]:
    print(f"  {w:<16} {tb[w]:>4} -> {ta[w]:>4}  {'*** REGRESSION ***' if tb[w] != ta[w] else 'ok'}")

## 2 — Chapter titles

Corrected against the printed NCTB book. Ch12 keeps the textbook's `জৈব অভিব্যক্তি` as
canonical with the more modern `বিবর্তন` as an alias, since the KG should say what the
curriculum says. Ch1 and ch6 carry aliases for the spelling variants.

In [ ]:
TITLES = {
    1:  ("জীবন পাঠ",                        "জীবনপাঠ"),
    2:  ("জীবকোষ ও টিস্যু",                 None),
    3:  ("কোষ বিভাজন",                      None),
    4:  ("জীবনীশক্তি",                      None),
    5:  ("খাদ্য, পুষ্টি ও পরিপাক",           None),
    6:  ("জীবে পরিবহন",                     "জীবে পরিবহণ"),
    7:  ("গ্যাসীয় বিনিময়",                  None),
    8:  ("রেচন প্রক্রিয়া",                   None),
    9:  ("দৃঢ়তা প্রদান ও চলন",              None),
    10: ("সমন্বয়",                          None),
    11: ("জীবের প্রজনন",                    None),
    12: ("জীবের বংশগতি ও জৈব অভিব্যক্তি",   "জীবের বংশগতি ও বিবর্তন"),
    13: ("জীবের পরিবেশ",                    None),
    14: ("জীবপ্রযুক্তি",                     None),
}

df["chapter_title_raw"] = df["chapter_title"]
df["chapter_title"] = df.chapter_no.map(lambda c: TITLES[c][0])
df["chapter_title_alt"] = df.chapter_no.map(lambda c: TITLES[c][1])

changed = (df.groupby("chapter_no")
             .agg(was=("chapter_title_raw", "first"), now=("chapter_title", "first")))
changed[changed.was != changed.now]

## 3 — Drop junk chunks

In [ ]:
def bengali_ratio(s):
    letters = [c for c in s if c.isalnum()]
    return sum(1 for c in letters if "ঀ" <= c <= "৿") / len(letters) if letters else 0.0

df["n_chars"] = df.text.str.len()
df["ben_ratio"] = df.text.map(bengali_ratio)
df["is_junk"] = (df.n_chars < 80) | (df.ben_ratio < 0.5)

clean = df[~df.is_junk].copy()
print(f"kept {len(clean)} of {len(df)} chunks ({len(clean)/len(df)*100:.1f}%)")
print(clean.groupby("chapter_no").size().rename("chunks").to_string())

## 4 — Export

In [ ]:
COLS = ["chunk_id", "class", "subject", "chapter_no", "chapter_title",
        "chapter_title_alt", "text", "was_repaired", "n_chars"]
out = clean[COLS].sort_values(["chapter_no", "chunk_id"]).reset_index(drop=True)
out.to_parquet(WORK / "biology_9_10_clean.parquet", index=False)

print(out.shape)
out.head(3)